# TTF Natural Gas Options — Interactive Jupyter Dashboard

**Sections : 1 — Pricer · 2 — Structures**

This notebook replaces the Streamlit dashboard with an interactive
Jupyter interface built on `ipywidgets`. Pricing functions are imported
directly from `black76_ttf.py` placed in the **same folder** as this
notebook.

**Requirements :**
```
pip install ipywidgets numpy scipy pandas
# JupyterLab ≥ 3.0 : widgets work out of the box
# Classic Notebook  : jupyter nbextension enable --py widgetsnbextension
```

Run the two code cells below ; the price card will update live as you
move the sliders.


In [ ]:
import math
import os
import sys
from pathlib import Path

# Make TTF modules importable no matter where the notebook is launched from
_HERE = Path(globals().get("__vsc_ipynb_file__", os.getcwd())).resolve().parent
if str(_HERE) not in sys.path:
    sys.path.append(str(_HERE))
# Fallback : also append the current working directory
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import ipywidgets as w
import pandas as pd
from IPython.display import display, HTML, clear_output

from black76_ttf import (
    b76_call, b76_put, b76_greeks,
    bach_call, bach_put, bach_greeks,
    b76_implied_vol, bach_implied_vol,
)

print("Imports OK — ready to build the Pricer UI.")


## Section 1 — Pricer

Move the sliders on the left ; the right panel refreshes with the
Call / Put prices for the selected model and the full table of Greeks.
The bottom table compares Black-76 and Bachelier side by side.


In [ ]:
# ── Widgets ────────────────────────────────────────────────────────────
F_w      = w.FloatSlider(value=30.0,  min=-20.0, max=250.0, step=0.5,
                         description="Forward F",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".1f")
K_w      = w.FloatSlider(value=30.0,  min=0.0,   max=250.0, step=0.5,
                         description="Strike K",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".1f")
Tdays_w  = w.IntSlider  (value=90,    min=1,     max=1825,  step=1,
                         description="T (days)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"))
rpct_w   = w.FloatSlider(value=2.0,   min=0.0,   max=10.0,  step=0.05,
                         description="Rate (%)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".2f")
sigma_w  = w.FloatSlider(value=50.0,  min=1.0,   max=250.0, step=1.0,
                         description="σ B76 (%)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".0f")
sign_w   = w.FloatSlider(value=8.0,   min=0.1,   max=50.0,  step=0.1,
                         description="σₙ Bach (€/MWh)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".1f")
type_w   = w.RadioButtons(options=["call", "put"], value="call",
                          description="Type",
                          style={"description_width": "110px"},
                          layout=w.Layout(width="280px"))
model_w  = w.Dropdown    (options=["Black-76", "Bachelier"], value="Black-76",
                          description="Model",
                          style={"description_width": "110px"},
                          layout=w.Layout(width="280px"))

out = w.Output()


# ── Pricing + HTML rendering ───────────────────────────────────────────
_CSS = """
<style>
  .ttf-card     { font-family: -apple-system, Segoe UI, Roboto, sans-serif;
                  color:#e2e8f0; background:#161625;
                  border:1px solid #2a2a42; border-radius:10px;
                  padding:14px 18px; margin:4px 0; }
  .ttf-card h3  { margin:0 0 8px 0; color:#f1f5f9; font-size:14pt;
                  border-bottom:1px solid #3b82f6; padding-bottom:4px; }
  .ttf-price    { display:flex; gap:20px; margin:8px 0 14px 0; }
  .ttf-px       { flex:1; background:#0f0f1a; border:1px solid #2a2a42;
                  border-radius:8px; padding:10px 14px; }
  .ttf-px .lbl  { color:#94a3b8; font-size:9pt; text-transform:uppercase;
                  letter-spacing:0.05em; }
  .ttf-px .val  { color:#f1f5f9; font-size:16pt; font-weight:600;
                  margin-top:2px; }
  .ttf-greeks   { border-collapse:collapse; width:100%; font-size:10pt; }
  .ttf-greeks th, .ttf-greeks td
                { padding:6px 10px; border-bottom:1px solid #2a2a42;
                  text-align:right; }
  .ttf-greeks th{ color:#94a3b8; text-align:left; font-weight:500;
                  background:#0f0f1a; }
  .ttf-greeks tr:nth-child(even) td { background:#12121d; }
  .ttf-tag      { display:inline-block; padding:2px 8px; margin-left:6px;
                  font-size:9pt; border-radius:10px; background:#1d4ed8; color:#fff; }
  .ttf-tag.bach { background:#7c3aed; }
</style>
"""


def render_pricer(F, K, Tdays, rpct, sigma_pct, sigma_n, option_type, model):
    """Compute prices/Greeks for both models and render an HTML card."""
    T     = Tdays / 365.0
    r     = rpct / 100.0
    sigma = sigma_pct / 100.0

    # Both models
    b76_c = b76_call(F, K, T, r, sigma)
    b76_p = b76_put (F, K, T, r, sigma)
    bc    = bach_call(F, K, T, r, sigma_n)
    bp    = bach_put (F, K, T, r, sigma_n)
    g76   = b76_greeks (F, K, T, r, sigma,  option_type)
    gbch  = bach_greeks(F, K, T, r, sigma_n, option_type)

    # Primary model (selected)
    if model == "Black-76":
        primary_c, primary_p, gp = b76_c, b76_p, g76
        tag_class = ""
    else:
        primary_c, primary_p, gp = bc, bp, gbch
        tag_class = "bach"

    # Put-call parity check
    df_disc = math.exp(-r * T)
    pcp_rhs = df_disc * (F - K)
    pcp_b76 = b76_c - b76_p
    pcp_bch = bc - bp

    moneyness = ("ATM" if abs(F - K) / max(F, 1e-6) < 0.005
                 else ("ITM" if (F > K) == (option_type == "call") else "OTM"))

    html = _CSS + f"""
    <div class="ttf-card">
      <h3>{model} — {option_type.upper()}
          <span class="ttf-tag {tag_class}">{model}</span>
          <span class="ttf-tag" style="background:#475569;">{moneyness}</span>
      </h3>
      <div class="ttf-price">
        <div class="ttf-px">
          <div class="lbl">Call price</div>
          <div class="val">{primary_c:.4f} €/MWh</div>
        </div>
        <div class="ttf-px">
          <div class="lbl">Put price</div>
          <div class="val">{primary_p:.4f} €/MWh</div>
        </div>
        <div class="ttf-px">
          <div class="lbl">Intrinsic ({option_type})</div>
          <div class="val">{(max(F-K,0) if option_type=='call' else max(K-F,0)):.4f}</div>
        </div>
      </div>

      <table class="ttf-greeks">
        <tr><th>Greek</th><th>Value</th><th>Unit</th></tr>
        <tr><td>Δ Delta</td>     <td>{gp.delta:+.4f}</td> <td>per unit F</td></tr>
        <tr><td>Γ Gamma</td>     <td>{gp.gamma:.6f}</td>   <td>per (€/MWh)²</td></tr>
        <tr><td>ν Vega</td>      <td>{gp.vega:.4f}</td>    <td>per unit vol (÷100 for 1 vol pt)</td></tr>
        <tr><td>Θ Theta/day</td> <td>{gp.theta:+.4f}</td>  <td>€/MWh / day</td></tr>
        <tr><td>ρ Rho</td>       <td>{gp.rho:+.6f}</td>    <td>per 1 pp change in r</td></tr>
        <tr><td>Vanna</td>       <td>{gp.vanna:+.6f}</td>  <td>∂Δ/∂σ = ∂ν/∂F</td></tr>
        <tr><td>Volga</td>       <td>{gp.volga:+.6f}</td>  <td>∂²V/∂σ²</td></tr>
      </table>
    </div>

    <div class="ttf-card">
      <h3>Side-by-side — Black-76 vs Bachelier</h3>
      <table class="ttf-greeks">
        <tr><th>Side</th><th>Black-76 price</th><th>Bachelier price</th>
            <th>Δ B76</th><th>Δ Bach</th>
            <th>ν B76</th><th>ν Bach</th></tr>
        <tr><td>Call</td>
            <td>{b76_c:.4f}</td><td>{bc:.4f}</td>
            <td>{g76.delta:+.4f}</td><td>{gbch.delta:+.4f}</td>
            <td>{g76.vega:.4f}</td><td>{gbch.vega:.4f}</td></tr>
        <tr><td>Put</td>
            <td>{b76_p:.4f}</td><td>{bp:.4f}</td>
            <td>{b76_greeks(F,K,T,r,sigma,'put').delta:+.4f}</td>
            <td>{bach_greeks(F,K,T,r,sigma_n,'put').delta:+.4f}</td>
            <td>{g76.vega:.4f}</td><td>{gbch.vega:.4f}</td></tr>
      </table>
      <p style="color:#94a3b8; font-size:9pt; margin-top:10px;">
        Put-call parity check &nbsp;·&nbsp;
        e<sup>-rT</sup>(F-K) = <b>{pcp_rhs:.6f}</b>
        &nbsp;·&nbsp; B76 C-P = {pcp_b76:.6f}
        &nbsp;·&nbsp; Bach C-P = {pcp_bch:.6f}
      </p>
    </div>
    """
    with out:
        clear_output(wait=True)
        display(HTML(html))


# ── Wire widgets → pricing ─────────────────────────────────────────────
widget_dict = {
    "F": F_w, "K": K_w, "Tdays": Tdays_w, "rpct": rpct_w,
    "sigma_pct": sigma_w, "sigma_n": sign_w,
    "option_type": type_w, "model": model_w,
}
_ = w.interactive_output(render_pricer, widget_dict)

# ── Layout ─────────────────────────────────────────────────────────────
left_col  = w.VBox([F_w, K_w, Tdays_w, rpct_w, sigma_w, sign_w])
right_col = w.VBox([type_w, model_w])
controls  = w.HBox([left_col, right_col])
display(controls, out)


## Section 2 — Structures

Price any of **10 classic multi-leg structures** on TTF futures
(Black-76 across all legs) and visualise the P&L at expiry.

Choose a structure in the dropdown → the matching strike inputs light
up → the P&L chart and metrics refresh live.

| Structure | Strikes | Extra |
|---|---|---|
| Straddle | K (= K1) | — |
| Strangle | K_put = K1, K_call = K2 | — |
| Bull call spread | K_lo = K1, K_hi = K2 | — |
| Bear put spread | K_lo = K1, K_hi = K2 | — |
| Butterfly | K_lo = K1, K_mid = K2, K_hi = K3 | — |
| Condor | K1, K2, K3, K4 | — |
| Collar | K_put = K1, K_call = K2 | — |
| Risk reversal | K_put = K1, K_call = K2 | — |
| Calendar spread | K = K1 | T_near, T_far (days) |
| Ratio spread | K_lo = K1, K_hi = K2 | Ratio (short qty) |


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

import structures_ttf as structs_mod

# Dark theme for matplotlib
plt.rcParams.update({
    "figure.facecolor":   "#0f0f1a",
    "axes.facecolor":     "#0f0f1a",
    "savefig.facecolor":  "#0f0f1a",
    "axes.edgecolor":     "#2a2a42",
    "axes.labelcolor":    "#cbd5e1",
    "xtick.color":        "#94a3b8",
    "ytick.color":        "#94a3b8",
    "text.color":         "#e2e8f0",
    "grid.color":         "#1e1e32",
    "font.size":          10,
})

_STRUCT_SPECS = {
    "Straddle":         {"strikes": 1, "extra": 0},
    "Strangle":         {"strikes": 2, "extra": 0},
    "Bull call spread": {"strikes": 2, "extra": 0},
    "Bear put spread":  {"strikes": 2, "extra": 0},
    "Butterfly":        {"strikes": 3, "extra": 0},
    "Condor":           {"strikes": 4, "extra": 0},
    "Collar":           {"strikes": 2, "extra": 0},
    "Risk reversal":    {"strikes": 2, "extra": 0},
    "Calendar spread":  {"strikes": 1, "extra": 2},  # T_near, T_far
    "Ratio spread":     {"strikes": 2, "extra": 1},  # ratio
}


def _dispatch(struct_name, F, T, r, sigma, strikes, extras):
    """Route widget values to the right structures_ttf function."""
    K1, K2, K3, K4 = strikes
    if struct_name == "Straddle":
        return structs_mod.straddle(F, K=K1, T=T, r=r, sigma=sigma)
    if struct_name == "Strangle":
        return structs_mod.strangle(F, K_put=K1, K_call=K2, T=T, r=r, sigma=sigma)
    if struct_name == "Bull call spread":
        return structs_mod.bull_call_spread(F, K_lo=K1, K_hi=K2, T=T, r=r, sigma=sigma)
    if struct_name == "Bear put spread":
        return structs_mod.bear_put_spread(F, K_lo=K1, K_hi=K2, T=T, r=r, sigma=sigma)
    if struct_name == "Butterfly":
        return structs_mod.butterfly(F, K_lo=K1, K_mid=K2, K_hi=K3,
                                     T=T, r=r, sigma=sigma)
    if struct_name == "Condor":
        return structs_mod.condor(F, K1=K1, K2=K2, K3=K3, K4=K4,
                                  T=T, r=r, sigma=sigma)
    if struct_name == "Collar":
        return structs_mod.collar(F, K_put=K1, K_call=K2, T=T, r=r, sigma=sigma)
    if struct_name == "Risk reversal":
        return structs_mod.risk_reversal(F, K_put=K1, K_call=K2, T=T, r=r, sigma=sigma)
    if struct_name == "Calendar spread":
        return structs_mod.calendar_spread(
            F, K=K1,
            T_far=extras[1] / 365.0,
            T_near=extras[0] / 365.0,
            r=r, sigma=sigma,
        )
    if struct_name == "Ratio spread":
        return structs_mod.ratio_spread(
            F, K_lo=K1, K_hi=K2, T=T, r=r, sigma=sigma,
            ratio=int(extras[0]),
        )
    raise ValueError(f"Unknown structure: {struct_name}")


def _validate(struct_name, strikes, extras):
    K1, K2, K3, K4 = strikes
    if struct_name == "Strangle" and K1 >= K2:
        return "K_put must be < K_call."
    if struct_name in ("Bull call spread", "Bear put spread", "Ratio spread")             and K1 >= K2:
        return "K_lo must be < K_hi."
    if struct_name == "Butterfly" and not (K1 < K2 < K3):
        return "Require K_lo < K_mid < K_hi."
    if struct_name == "Condor" and not (K1 < K2 < K3 < K4):
        return "Require K1 < K2 < K3 < K4."
    if struct_name in ("Collar", "Risk reversal") and K1 >= K2:
        return "K_put must be < K_call."
    if struct_name == "Calendar spread" and extras[1] <= extras[0]:
        return "T_far must be > T_near."
    return None


# ── Widgets ────────────────────────────────────────────────────────────
struct_dd = w.Dropdown(
    options=list(_STRUCT_SPECS.keys()),
    value="Straddle",
    description="Structure",
    style={"description_width": "110px"},
    layout=w.Layout(width="380px"),
)

F_s    = w.FloatSlider(value=30.0, min=-20, max=250, step=0.5,
                        description="Forward F",
                        style={"description_width": "110px"},
                        layout=w.Layout(width="380px"),
                        readout_format=".1f")
T_s    = w.IntSlider  (value=90, min=1, max=1825,
                        description="T (days)",
                        style={"description_width": "110px"},
                        layout=w.Layout(width="380px"))
r_s    = w.FloatSlider(value=2.0, min=0, max=10, step=0.05,
                        description="Rate (%)",
                        style={"description_width": "110px"},
                        layout=w.Layout(width="380px"),
                        readout_format=".2f")
sig_s  = w.FloatSlider(value=50.0, min=1, max=250, step=1,
                        description="σ (%)",
                        style={"description_width": "110px"},
                        layout=w.Layout(width="380px"),
                        readout_format=".0f")

K1_s = w.FloatSlider(value=30.0, min=0.1, max=250, step=0.5,
                      description="K1",
                      style={"description_width": "110px"},
                      layout=w.Layout(width="380px"),
                      readout_format=".1f")
K2_s = w.FloatSlider(value=33.0, min=0.1, max=250, step=0.5,
                      description="K2",
                      style={"description_width": "110px"},
                      layout=w.Layout(width="380px"),
                      readout_format=".1f")
K3_s = w.FloatSlider(value=36.0, min=0.1, max=250, step=0.5,
                      description="K3",
                      style={"description_width": "110px"},
                      layout=w.Layout(width="380px"),
                      readout_format=".1f")
K4_s = w.FloatSlider(value=39.0, min=0.1, max=250, step=0.5,
                      description="K4",
                      style={"description_width": "110px"},
                      layout=w.Layout(width="380px"),
                      readout_format=".1f")

ex1_s = w.IntSlider  (value=30,  min=1, max=1825,
                      description="T_near (days)",
                      style={"description_width": "110px"},
                      layout=w.Layout(width="380px"))
ex2_s = w.IntSlider  (value=180, min=1, max=1825,
                      description="T_far (days)",
                      style={"description_width": "110px"},
                      layout=w.Layout(width="380px"))
ratio_s = w.IntSlider(value=2, min=1, max=10,
                      description="Ratio",
                      style={"description_width": "110px"},
                      layout=w.Layout(width="380px"))

structs_out = w.Output()


def _set_visibility(struct_name):
    """Show only the K/extra widgets relevant for the chosen structure."""
    spec = _STRUCT_SPECS[struct_name]
    K_widgets = [K1_s, K2_s, K3_s, K4_s]
    for i, kw in enumerate(K_widgets):
        kw.layout.display = "" if i < spec["strikes"] else "none"

    # Custom labels per structure
    label_map = {
        "Straddle":         ["K"],
        "Strangle":         ["K_put", "K_call"],
        "Bull call spread": ["K_lo", "K_hi"],
        "Bear put spread":  ["K_lo", "K_hi"],
        "Butterfly":        ["K_lo", "K_mid", "K_hi"],
        "Condor":           ["K1", "K2", "K3", "K4"],
        "Collar":           ["K_put", "K_call"],
        "Risk reversal":    ["K_put", "K_call"],
        "Calendar spread":  ["K"],
        "Ratio spread":     ["K_lo", "K_hi"],
    }
    labels = label_map[struct_name]
    for i, kw in enumerate(K_widgets[:len(labels)]):
        kw.description = labels[i]

    # Extra widgets (calendar / ratio)
    ex1_s.layout.display = "none"
    ex2_s.layout.display = "none"
    ratio_s.layout.display = "none"
    if struct_name == "Calendar spread":
        ex1_s.layout.display = ""
        ex2_s.layout.display = ""
    elif struct_name == "Ratio spread":
        ratio_s.layout.display = ""


def _render_structure(*_args, **_kwargs):
    struct_name = struct_dd.value
    _set_visibility(struct_name)

    F = F_s.value; T = T_s.value / 365.0; r = r_s.value / 100.0; sigma = sig_s.value / 100.0
    strikes = (K1_s.value, K2_s.value, K3_s.value, K4_s.value)
    extras  = (ex1_s.value, ex2_s.value) if struct_name == "Calendar spread" else (ratio_s.value,)

    err = _validate(struct_name, strikes, extras)

    with structs_out:
        clear_output(wait=True)
        if err:
            display(HTML(f"<div style='color:#f87171; font-weight:600; "
                         f"padding:10px;'>⚠ {err}</div>"))
            return

        try:
            res = _dispatch(struct_name, F, T, r, sigma, strikes, extras)
        except Exception as exc:
            display(HTML(f"<div style='color:#f87171;'>Pricing error: {exc}</div>"))
            return

        # Header card
        premium_label = "Debit" if res.price >= 0 else "Credit"
        if res.max_profit == math.inf:
            mp_txt = "∞"
        else:
            mp_txt = f"{res.max_profit:+.4f}"
        if res.max_loss == -math.inf:
            ml_txt = "−∞"
        else:
            ml_txt = f"{res.max_loss:+.4f}"
        be_txt = ", ".join(f"{b:.3f}" for b in res.breakevens) if res.breakevens else "—"

        html = f"""
        <div class="ttf-card">
          <h3>{res.name} — {len(res.legs)} legs</h3>
          <div class="ttf-price">
            <div class="ttf-px"><div class="lbl">Net premium ({premium_label})</div>
                 <div class="val">{res.price:+.4f} €/MWh</div></div>
            <div class="ttf-px"><div class="lbl">Max profit</div>
                 <div class="val">{mp_txt}</div></div>
            <div class="ttf-px"><div class="lbl">Max loss</div>
                 <div class="val">{ml_txt}</div></div>
            <div class="ttf-px"><div class="lbl">Breakevens</div>
                 <div class="val" style="font-size:11pt;">{be_txt}</div></div>
          </div>
          <table class="ttf-greeks">
            <tr><th>Net Δ</th><th>Net Γ</th><th>Net ν</th><th>Net Θ/day</th></tr>
            <tr><td>{res.delta:+.4f}</td><td>{res.gamma:+.6f}</td>
                <td>{res.vega:+.4f}</td><td>{res.theta:+.4f}</td></tr>
          </table>
        </div>
        """
        display(HTML(html))

        # Legs detail
        rows = []
        for i, leg in enumerate(res.legs, 1):
            rows.append({
                "#": i,
                "Side": "Long" if leg.sign > 0 else "Short",
                "Type": leg.option_type,
                "Qty": leg.qty,
                "Strike": round(leg.K, 4),
                "T (y)": round(leg.T, 4),
                "σ": round(leg.sigma, 4),
                "Unit price": round(leg.unit_price, 6),
                "Unit Δ": round(leg.unit_delta, 6),
                "Unit ν": round(leg.unit_vega, 6),
            })
        display(pd.DataFrame(rows))

        # Matplotlib P&L chart
        F_T = np.array([pt[0] for pt in res.pnl_at_expiry])
        pnl = np.array([pt[1] for pt in res.pnl_at_expiry])

        fig, ax = plt.subplots(figsize=(9, 4.2))
        ax.plot(F_T, pnl, color="#3b82f6", lw=2.2)

        # Shade profit (green) / loss (red)
        ax.fill_between(F_T, pnl, 0, where=(pnl >= 0),
                         color="#22c55e", alpha=0.22, interpolate=True,
                         label="Profit")
        ax.fill_between(F_T, pnl, 0, where=(pnl < 0),
                         color="#ef4444", alpha=0.22, interpolate=True,
                         label="Loss")

        ax.axhline(0, color="#64748b", lw=1, ls="--")
        ax.axvline(F, color="#facc15", lw=1, ls=":")
        ax.text(F, ax.get_ylim()[1]*0.9, f"  F = {F:.1f}",
                 color="#facc15", fontsize=9, va="top")

        for be in res.breakevens:
            ax.axvline(be, color="#f87171", lw=1, ls=":")
            ax.text(be, ax.get_ylim()[0]*0.9, f"BE {be:.2f}",
                     color="#f87171", fontsize=8, va="bottom", ha="right",
                     rotation=90)

        ax.set_xlabel("Forward at expiry $F_T$ (EUR/MWh)")
        ax.set_ylabel("P&L (EUR/MWh)")
        ax.set_title(f"{res.name} — P&L at expiry", color="#f1f5f9", fontsize=12)
        ax.grid(True, alpha=0.35)
        ax.legend(loc="best", facecolor="#161625", edgecolor="#2a2a42",
                  labelcolor="#cbd5e1", framealpha=0.9)
        plt.tight_layout()
        plt.show()


# Wire up observers
for widget in (struct_dd, F_s, T_s, r_s, sig_s,
               K1_s, K2_s, K3_s, K4_s, ex1_s, ex2_s, ratio_s):
    widget.observe(_render_structure, names="value")

_set_visibility("Straddle")   # initial layout
_render_structure()           # initial render

# Display widgets
left_col  = w.VBox([struct_dd, F_s, T_s, r_s, sig_s])
right_col = w.VBox([K1_s, K2_s, K3_s, K4_s, ex1_s, ex2_s, ratio_s])
display(w.HBox([left_col, right_col]), structs_out)


## Section 3 — Vol Surface

Interactive 3D visualisation of the TTF implied-volatility surface.

**Data sources :**
* *Pre-computed* — loads `ttf_output/ttf_vol_surface.json`, the file
  last exported by `ttf_market_data.py`.
* *Live rebuild* — constructs a fresh surface via
  `ttf_market_data.VolatilitySurfaceBuilder`, re-scaling the built-in
  ATM vols so the 3-month pillar equals the **σ slider in Section 1**.

Use the moneyness / maturity sliders to zoom on a specific region of
the surface ; the matplotlib 3D plot redraws live.


In [ ]:
import json
from pathlib import Path
from datetime import date

import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3D projection)
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap

_VOL_JSON = _HERE / "ttf_output" / "ttf_vol_surface.json"


def _load_precomputed():
    try:
        with open(_VOL_JSON) as fh:
            j = json.load(fh)
        return pd.DataFrame(j["surface"]), j["reference_date"]
    except (FileNotFoundError, json.JSONDecodeError):
        return None, None


def _load_live(ref_date_iso: str, atm_target: float):
    """Rebuild surface with ATM vols scaled to the target 3M ATM."""
    from ttf_market_data import TTFForwardCurve, VolatilitySurfaceBuilder
    ref = date.fromisoformat(ref_date_iso)
    fwd = TTFForwardCurve(ref).load()
    builder = VolatilitySurfaceBuilder(fwd, ref)
    base = dict(builder._ATM_VOLS)
    scale = atm_target / base[3/12]
    surface = builder.build(atm_vols={t: v * scale for t, v in base.items()})
    return surface.to_dataframe(), str(ref)


# ── Widgets ───────────────────────────────────────────────────────────
source_w = w.RadioButtons(
    options=["Pre-computed (ttf_output/)", "Live rebuild"],
    value="Pre-computed (ttf_output/)",
    description="Source",
    style={"description_width": "110px"},
    layout=w.Layout(width="400px"),
)

ref_w = w.Text(
    value=str(date.today()),
    description="Ref date",
    style={"description_width": "110px"},
    layout=w.Layout(width="260px"),
)

mony_w = w.FloatRangeSlider(
    value=[0.5, 1.8], min=0.3, max=3.0, step=0.05,
    description="K/F range",
    style={"description_width": "110px"},
    layout=w.Layout(width="400px"),
    readout_format=".2f",
)

maxT_w = w.FloatSlider(
    value=2.0, min=0.25, max=3.0, step=0.25,
    description="Max T (y)",
    style={"description_width": "110px"},
    layout=w.Layout(width="400px"),
    readout_format=".2f",
)

elev_w = w.IntSlider(
    value=25, min=0, max=90, step=5,
    description="Elevation°",
    style={"description_width": "110px"},
    layout=w.Layout(width="400px"),
)
azim_w = w.IntSlider(
    value=-135, min=-180, max=180, step=5,
    description="Azimuth°",
    style={"description_width": "110px"},
    layout=w.Layout(width="400px"),
)

vol_out = w.Output()

# Custom "cool→hot" colormap mirroring the Streamlit / Plotly version
_TTF_CMAP = LinearSegmentedColormap.from_list(
    "ttf_vol",
    ["#1e3a5f", "#2563eb", "#7c3aed", "#db2777", "#ef4444"],
)


def _surface_to_grid(df):
    """Pivot long DataFrame → (Ts, Ks, Z) with NaN-row interpolation."""
    Ts = sorted(df["T"].unique())
    Ks = sorted(df["strike"].unique())
    Z = np.full((len(Ts), len(Ks)), np.nan)
    for i, t in enumerate(Ts):
        slice_ = df[df["T"] == t].set_index("strike")["vol"]
        for j, k in enumerate(Ks):
            if k in slice_.index:
                Z[i, j] = float(slice_.loc[k])
    for i in range(len(Ts)):
        mask = ~np.isnan(Z[i])
        if mask.sum() >= 2:
            Z[i] = np.interp(Ks, np.array(Ks)[mask], Z[i][mask])
    return np.array(Ts), np.array(Ks), Z


def _render_surface(*_args, **_kwargs):
    with vol_out:
        clear_output(wait=True)

        if source_w.value.startswith("Pre-computed"):
            df, ref_iso = _load_precomputed()
            if df is None:
                display(HTML("<div style='color:#f87171;'>"
                             "⚠ ttf_output/ttf_vol_surface.json not found — "
                             "run `python ttf_market_data.py` first, or switch "
                             "to Live rebuild."
                             "</div>"))
                return
        else:
            try:
                # Reuse the σ slider from Section 1 if present, else default 50%
                atm_target = sigma_w.value / 100.0 if "sigma_w" in globals() else 0.50
                df, ref_iso = _load_live(ref_w.value, atm_target)
            except Exception as exc:
                display(HTML(f"<div style='color:#f87171;'>Live rebuild failed : "
                             f"{exc}</div>"))
                return

        # Filter
        df2 = df[df["T"] <= maxT_w.value].copy()
        df2 = df2[df2.apply(lambda r: mony_w.value[0] <= r["strike"] / r["F"]
                             <= mony_w.value[1], axis=1)]
        if df2.empty:
            display(HTML("<div style='color:#f87171;'>No surface points in the "
                         "selected range — loosen the filters.</div>"))
            return

        Ts, Ks, Z = _surface_to_grid(df2)

        # Header card
        display(HTML(f"""
        <div class="ttf-card">
          <h3>TTF Implied Volatility Surface</h3>
          <div style="color:#94a3b8; font-size:10pt;">
            Reference date : <b>{ref_iso}</b>
            &nbsp;·&nbsp; Source : {source_w.value}
            &nbsp;·&nbsp; {len(Ts)} maturities × {len(Ks)} strikes
            &nbsp;·&nbsp; σ range : {Z.min()*100:.1f}% — {Z.max()*100:.1f}%
          </div>
        </div>"""))

        # 3D surface
        T_mesh, K_mesh = np.meshgrid(Ts, Ks, indexing="ij")

        fig = plt.figure(figsize=(10, 6.5))
        ax = fig.add_subplot(111, projection="3d")
        surf = ax.plot_surface(
            K_mesh, T_mesh, Z * 100.0,
            cmap=_TTF_CMAP, edgecolor="#1e1e32", linewidth=0.15,
            antialiased=True, alpha=0.92,
        )

        ax.set_xlabel("Strike (EUR/MWh)", color="#cbd5e1", labelpad=8)
        ax.set_ylabel("Tenor (years)",    color="#cbd5e1", labelpad=8)
        ax.set_zlabel("σ (%)",            color="#cbd5e1", labelpad=4)

        ax.set_title("TTF implied-volatility surface",
                     color="#f1f5f9", fontsize=12, pad=14)

        ax.tick_params(colors="#94a3b8")
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.pane.fill = False
            axis.pane.set_edgecolor("#2a2a42")
            axis._axinfo["grid"]["color"] = (0.17, 0.17, 0.26, 1.0)

        ax.view_init(elev=elev_w.value, azim=azim_w.value)

        cbar = fig.colorbar(surf, ax=ax, shrink=0.62, pad=0.08)
        cbar.set_label("σ (%)", color="#cbd5e1")
        cbar.ax.yaxis.set_tick_params(color="#94a3b8")
        plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="#94a3b8")

        plt.tight_layout()
        plt.show()

        # Smile cross-sections (2D)
        fig2, ax2 = plt.subplots(figsize=(9, 3.4))
        palette = ["#60a5fa", "#a78bfa", "#34d399", "#f87171",
                   "#facc15", "#fb923c", "#e879f9", "#2dd4bf"]
        for idx, t in enumerate(Ts):
            sub = df2[df2["T"] == t].sort_values("strike")
            if sub.empty:
                continue
            label = f"{t:.2f}y" if t >= 0.5 else f"{int(round(t*12))}M"
            ax2.plot(sub["strike"], sub["vol"] * 100.0,
                     marker="o", ms=4, lw=2,
                     color=palette[idx % len(palette)], label=label)
        ax2.set_xlabel("Strike (EUR/MWh)")
        ax2.set_ylabel("σ (%)")
        ax2.set_title("Smile cross-sections", color="#f1f5f9", fontsize=11)
        ax2.grid(True, alpha=0.3)
        ax2.legend(facecolor="#161625", edgecolor="#2a2a42",
                   labelcolor="#cbd5e1", framealpha=0.9, fontsize=8,
                   loc="best", ncol=2)
        plt.tight_layout()
        plt.show()


for widget in (source_w, ref_w, mony_w, maxT_w, elev_w, azim_w):
    widget.observe(_render_surface, names="value")

_render_surface()   # initial render

left  = w.VBox([source_w, ref_w, mony_w, maxT_w])
right = w.VBox([elev_w, azim_w])
display(w.HBox([left, right]), vol_out)


## Section 4 — Spread TTF/HH (Margrabe)

Price a European option on the **TTF − Henry Hub basis** using
Margrabe's (1978) exchange-option formula.

* The TTF forward is quoted in **EUR/MWh** and converted internally to
  **USD/MMBtu** via `F_TTF_usd = F_TTF_eur × FX / 3.412142`.
* Henry Hub is already quoted in **USD/MMBtu**.
* The spread vol is σ_spread = √(σ_TTF² + σ_HH² − 2ρσ_TTF σ_HH) —
  higher correlation ⇒ *lower* spread vol ⇒ *cheaper* option.
* *Implied correlation* inverts the market price to find the ρ that
  matches the quote.


In [ ]:
import ttf_hh_spread as sp

# ── Widgets ───────────────────────────────────────────────────────────
F_ttf_eur_w = w.FloatSlider(value=30.0, min=5.0, max=120.0, step=0.5,
                            description="F TTF (€/MWh)",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"),
                            readout_format=".2f")
F_hh_w      = w.FloatSlider(value=3.0,  min=0.5, max=20.0, step=0.05,
                            description="F HH ($/MMBtu)",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"),
                            readout_format=".2f")
fx_w        = w.FloatSlider(value=1.08, min=0.80, max=1.40, step=0.005,
                            description="FX EUR/USD",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"),
                            readout_format=".4f")

sigma_ttf_w = w.FloatSlider(value=60.0, min=5.0, max=200.0, step=1.0,
                            description="σ TTF (%)",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"),
                            readout_format=".0f")
sigma_hh_w  = w.FloatSlider(value=50.0, min=5.0, max=200.0, step=1.0,
                            description="σ HH (%)",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"),
                            readout_format=".0f")
rho_w       = w.FloatSlider(value=0.35, min=-0.95, max=0.95, step=0.01,
                            description="Correlation ρ",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"),
                            readout_format=".2f")

T_spread_w  = w.IntSlider  (value=180, min=7, max=730, step=1,
                            description="T (days)",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"))
r_usd_w     = w.FloatSlider(value=4.5, min=0.0, max=10.0, step=0.05,
                            description="Rate USD (%)",
                            style={"description_width": "130px"},
                            layout=w.Layout(width="400px"),
                            readout_format=".2f")
sp_type_w   = w.RadioButtons(options=["call", "put"], value="call",
                             description="Type",
                             style={"description_width": "130px"})
mkt_px_w    = w.FloatText(value=0.0, step=0.01,
                          description="Market price ($/MMBtu)",
                          style={"description_width": "170px"},
                          layout=w.Layout(width="340px"))

spread_out = w.Output()


def _render_spread(*_args, **_kwargs):
    F_ttf_eur = F_ttf_eur_w.value
    F_hh      = F_hh_w.value
    fx        = fx_w.value
    sigma_ttf = sigma_ttf_w.value / 100.0
    sigma_hh  = sigma_hh_w.value  / 100.0
    rho       = rho_w.value
    T         = T_spread_w.value / 365.0
    r_usd     = r_usd_w.value   / 100.0
    opt_type  = sp_type_w.value
    mkt_px    = mkt_px_w.value

    F_ttf_usd = sp.ttf_eur_to_usd(F_ttf_eur, fx)

    res = sp.spread_price(F_ttf_eur, F_hh, fx, T, r_usd,
                           sigma_ttf, sigma_hh, rho, opt_type)

    # Put-call parity check
    df_disc = math.exp(-r_usd * T)
    pcp_rhs = df_disc * (F_ttf_usd - F_hh)
    call_p  = sp.margrabe_price(F_ttf_usd, F_hh, T, r_usd,
                                 sigma_ttf, sigma_hh, rho, "call")
    put_p   = sp.margrabe_price(F_ttf_usd, F_hh, T, r_usd,
                                 sigma_ttf, sigma_hh, rho, "put")

    with spread_out:
        clear_output(wait=True)

        # Header + inputs recap
        spread_usd = F_ttf_usd - F_hh
        spread_eur = sp.spread_usd_to_eur(spread_usd, fx)
        header = f"""
        <div class="ttf-card">
          <h3>Margrabe spread — {opt_type.upper()}</h3>
          <div class="ttf-price">
            <div class="ttf-px">
              <div class="lbl">TTF forward</div>
              <div class="val">{F_ttf_eur:.2f} €/MWh</div>
              <div class="lbl" style="margin-top:4px;">{F_ttf_usd:.4f} $/MMBtu (FX {fx:.4f})</div>
            </div>
            <div class="ttf-px">
              <div class="lbl">HH forward</div>
              <div class="val">{F_hh:.4f} $/MMBtu</div>
            </div>
            <div class="ttf-px">
              <div class="lbl">Spread TTF−HH</div>
              <div class="val">{spread_usd:+.4f} $/MMBtu</div>
              <div class="lbl" style="margin-top:4px;">{spread_eur:+.4f} €/MWh</div>
            </div>
            <div class="ttf-px">
              <div class="lbl">σ spread (Margrabe)</div>
              <div class="val">{res.sigma_spread*100:.2f}%</div>
              <div class="lbl" style="margin-top:4px;">
                √(σ²_TTF + σ²_HH − 2ρσ_TTF σ_HH)
              </div>
            </div>
          </div>
        </div>
        """
        display(HTML(header))

        # Price card + Greeks
        g = res.greeks
        card = f"""
        <div class="ttf-card">
          <h3>Option premium &amp; Greeks</h3>
          <div class="ttf-price">
            <div class="ttf-px">
              <div class="lbl">Premium</div>
              <div class="val">{res.price:.4f} $/MMBtu</div>
              <div class="lbl" style="margin-top:4px;">= {res.price_eur:.4f} €/MWh</div>
            </div>
            <div class="ttf-px">
              <div class="lbl">Δ TTF</div>
              <div class="val">{g.delta_ttf:+.4f}</div>
            </div>
            <div class="ttf-px">
              <div class="lbl">Δ HH</div>
              <div class="val">{g.delta_hh:+.4f}</div>
            </div>
            <div class="ttf-px">
              <div class="lbl">Γ TTF</div>
              <div class="val">{g.gamma_ttf:+.6f}</div>
            </div>
          </div>
          <table class="ttf-greeks">
            <tr><th>Vega TTF</th><th>Vega HH</th><th>Vega ρ</th><th>Θ /day</th></tr>
            <tr>
              <td>{g.vega_ttf:+.4f}</td>
              <td>{g.vega_hh:+.4f}</td>
              <td>{g.vega_rho:+.4f}</td>
              <td>{g.theta:+.4f}</td>
            </tr>
            <tr><td colspan="4" style="color:#94a3b8; font-size:9pt;">
              Vegas are "per unit vol" (1.0 = 100 vol-points). Divide by 100
              for the market-convention "per 1 vol-point".
            </td></tr>
          </table>
          <p style="color:#94a3b8; font-size:9pt; margin-top:10px;">
            Put-call parity check &nbsp;·&nbsp;
            C − P = <b>{call_p - put_p:.6f}</b>
            &nbsp;·&nbsp; e<sup>-rT</sup>(F_TTF − F_HH) =
            <b>{pcp_rhs:.6f}</b>
            &nbsp;·&nbsp; error = {abs(call_p - put_p - pcp_rhs):.2e}
          </p>
        </div>
        """
        display(HTML(card))

        # Implied correlation (optional — only if a market price is provided)
        if mkt_px > 0:
            try:
                impl_rho = sp.implied_correlation(
                    mkt_px, F_ttf_usd, F_hh, T, r_usd,
                    sigma_ttf, sigma_hh, opt_type,
                )
                sig_impl = math.sqrt(
                    max(sigma_ttf**2 + sigma_hh**2
                        - 2*impl_rho*sigma_ttf*sigma_hh, 0.0)
                )
                impl_card = f"""
                <div class="ttf-card">
                  <h3>Implied correlation from market quote</h3>
                  <div class="ttf-price">
                    <div class="ttf-px">
                      <div class="lbl">Market quote</div>
                      <div class="val">{mkt_px:.4f} $/MMBtu</div>
                    </div>
                    <div class="ttf-px">
                      <div class="lbl">Implied ρ</div>
                      <div class="val">{impl_rho:+.4f}</div>
                    </div>
                    <div class="ttf-px">
                      <div class="lbl">σ spread at impl ρ</div>
                      <div class="val">{sig_impl*100:.2f}%</div>
                    </div>
                    <div class="ttf-px">
                      <div class="lbl">Δ vs slider ρ</div>
                      <div class="val">{impl_rho - rho:+.4f}</div>
                    </div>
                  </div>
                </div>
                """
                display(HTML(impl_card))
            except Exception as exc:
                display(HTML(
                    f"<div class='ttf-card' style='color:#f87171;'>"
                    f"Implied-correlation solver : {exc}</div>"
                ))

        # ── Payoff chart at expiry ──────────────────────────────────────
        # Scan F_TTF (USD/MMBtu) around current level, keep F_HH fixed.
        lo = max(F_ttf_usd * 0.5, 0.1)
        hi = F_ttf_usd * 1.5
        F_grid = np.linspace(lo, hi, 201)

        if opt_type == "call":
            payoff   = np.maximum(F_grid - F_hh, 0.0)
            pnl_at   = payoff - res.price
            label    = "max(F_TTF − F_HH, 0)"
        else:
            payoff   = np.maximum(F_hh - F_grid, 0.0)
            pnl_at   = payoff - res.price
            label    = "max(F_HH − F_TTF, 0)"

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

        # LEFT : payoff
        ax1.plot(F_grid, payoff, color="#60a5fa", lw=2.2, label=label)
        ax1.axhline(0, color="#64748b", lw=1, ls="--")
        ax1.axvline(F_hh,      color="#22c55e", lw=1, ls=":",
                    label=f"F_HH = {F_hh:.2f}")
        ax1.axvline(F_ttf_usd, color="#facc15", lw=1, ls=":",
                    label=f"F_TTF = {F_ttf_usd:.2f}")
        ax1.set_xlabel("F_TTF at expiry (USD/MMBtu)")
        ax1.set_ylabel("Payoff (USD/MMBtu)")
        ax1.set_title(f"Spread payoff — {opt_type}", color="#f1f5f9", fontsize=11)
        ax1.grid(True, alpha=0.3)
        ax1.legend(facecolor="#161625", edgecolor="#2a2a42",
                   labelcolor="#cbd5e1", framealpha=0.9, fontsize=8)

        # RIGHT : P&L = payoff - premium
        ax2.plot(F_grid, pnl_at, color="#3b82f6", lw=2.2)
        ax2.fill_between(F_grid, pnl_at, 0, where=(pnl_at >= 0),
                          color="#22c55e", alpha=0.22)
        ax2.fill_between(F_grid, pnl_at, 0, where=(pnl_at < 0),
                          color="#ef4444", alpha=0.22)
        ax2.axhline(0, color="#64748b", lw=1, ls="--")
        ax2.axvline(F_hh,      color="#22c55e", lw=1, ls=":")
        ax2.axvline(F_ttf_usd, color="#facc15", lw=1, ls=":")

        # Breakeven : payoff - premium = 0
        try:
            if opt_type == "call":
                be = F_hh + res.price
                ax2.axvline(be, color="#f87171", lw=1, ls=":")
                ax2.text(be, ax2.get_ylim()[1]*0.85, f"BE {be:.2f}",
                         color="#f87171", fontsize=9, ha="right", rotation=90)
            else:
                be = F_hh - res.price
                if lo <= be <= hi:
                    ax2.axvline(be, color="#f87171", lw=1, ls=":")
                    ax2.text(be, ax2.get_ylim()[1]*0.85, f"BE {be:.2f}",
                             color="#f87171", fontsize=9, ha="right", rotation=90)
        except Exception:
            pass

        ax2.set_xlabel("F_TTF at expiry (USD/MMBtu)")
        ax2.set_ylabel("P&L (USD/MMBtu)")
        ax2.set_title("P&L = payoff − premium", color="#f1f5f9", fontsize=11)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


for widget in (F_ttf_eur_w, F_hh_w, fx_w,
               sigma_ttf_w, sigma_hh_w, rho_w,
               T_spread_w, r_usd_w, sp_type_w, mkt_px_w):
    widget.observe(_render_spread, names="value")

_render_spread()

left  = w.VBox([F_ttf_eur_w, F_hh_w, fx_w, T_spread_w, r_usd_w])
right = w.VBox([sigma_ttf_w, sigma_hh_w, rho_w, sp_type_w, mkt_px_w])
display(w.HBox([left, right]), spread_out)


## Section 5 — Expiry Dates

Next monthly TTF option-expiry dates with time-to-expiry under **two
conventions** that co-exist in the project :

* **ICE Endex (official)** — `ttf_expiry_date` : 5 calendar days before
  the 1st of the delivery month, rolled back to the previous business
  day if needed, then rolled back one more BD if it matches the futures
  LTD. Uses the NL + UK holiday calendar.
* **Legacy (5 BD before futures LTD)** — `options_expiry_from_delivery`
  : the original rule used by the pricing path (`t_from_contract`,
  `b76_price_ttf`, structures, spread).

The table also flags the futures LTD and the T in years (ACT/365,
start-of-day : today is included).


In [ ]:
from datetime import date, timedelta

from black76_ttf import (
    ttf_expiry_date, ttf_time_to_expiry, ttf_next_expiries,
    ttf_is_business_day,
    options_expiry_from_delivery, futures_expiry_from_delivery,
    t_from_delivery,
)

# ── Widgets ───────────────────────────────────────────────────────────
ref_date_w = w.DatePicker(
    value=date.today(),
    description="Reference date",
    style={"description_width": "140px"},
    layout=w.Layout(width="360px"),
)
n_w = w.IntSlider(
    value=12, min=1, max=36, step=1,
    description="# contracts",
    style={"description_width": "140px"},
    layout=w.Layout(width="360px"),
)

expiry_out = w.Output()

_MONTH_CODES = "FGHJKMNQUVXZ"


def _render_expiries(*_args, **_kwargs):
    ref = ref_date_w.value or date.today()
    n   = n_w.value

    rows = []
    year, month = ref.year, ref.month
    produced = 0
    # Walk forward month by month so every generated contract has
    # an expiry strictly after the reference date.
    while produced < n:
        ice_exp = ttf_expiry_date(month, year)
        if ice_exp > ref:
            legacy_exp = options_expiry_from_delivery(year, month)
            fut_ltd    = futures_expiry_from_delivery(year, month)
            code       = f"TTF{_MONTH_CODES[month-1]}{str(year)[-2:]}"
            label      = f"{date(year, month, 1).strftime('%b-%y')}"

            ice_days    = (ice_exp    - ref).days + 1  # today included
            legacy_days = (legacy_exp - ref).days + 1
            ice_T    = max(ice_days,    0) / 365.0
            legacy_T = max(legacy_days, 0) / 365.0

            rows.append({
                "Contract":     code,
                "Delivery":     label,
                "ICE Endex expiry":   str(ice_exp),
                "Legacy expiry":      str(legacy_exp),
                "Futures LTD":        str(fut_ltd),
                "Δ (ICE − Legacy)":   f"{(ice_exp - legacy_exp).days:+d} days",
                "T (ICE, years)":     round(ice_T, 4),
                "T (Legacy, years)":  round(legacy_T, 4),
                "Cal days (ICE)":     ice_days,
                "Trading day?":       "✓" if ttf_is_business_day(ice_exp) else "—",
            })
            produced += 1

        month += 1
        if month > 12:
            month = 1
            year += 1

    df = pd.DataFrame(rows)

    with expiry_out:
        clear_output(wait=True)

        # Header card
        header = f"""
        <div class="ttf-card">
          <h3>TTF option expiries — next {n} monthly contracts</h3>
          <div style="color:#94a3b8; font-size:10pt;">
            Reference date : <b>{ref}</b> · Convention : start-of-day
            (today included in T) · NL+UK holiday calendar
          </div>
        </div>
        """
        display(HTML(header))

        # Styled DataFrame (dark colours)
        styled = (
            df.style
              .format({"T (ICE, years)":    "{:.4f}",
                       "T (Legacy, years)": "{:.4f}"})
              .set_properties(**{
                  "background-color": "#161625",
                  "color":            "#cbd5e1",
                  "border":           "1px solid #2a2a42",
                  "padding":          "4px 10px",
              })
              .set_table_styles([
                  {"selector": "th",
                   "props": [("background-color", "#1e1e32"),
                             ("color",            "#f1f5f9"),
                             ("text-align",       "left"),
                             ("padding",          "6px 10px")]},
                  {"selector": "caption",
                   "props": [("color", "#94a3b8"),
                             ("font-size", "10pt"),
                             ("padding", "4px 0")]},
              ])
        )
        display(styled)

        # ── Timeline chart ─────────────────────────────────────────────
        if len(df) >= 2:
            fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(df) + 1.2)))
            y = np.arange(len(df))

            ice_dates    = [date.fromisoformat(d) for d in df["ICE Endex expiry"]]
            legacy_dates = [date.fromisoformat(d) for d in df["Legacy expiry"]]
            fut_dates    = [date.fromisoformat(d) for d in df["Futures LTD"]]

            ax.barh(y, [(f - ref).days for f in fut_dates],
                    color="#1e3a5f", edgecolor="#2a2a42",
                    height=0.7, alpha=0.55, label="Futures LTD")
            ax.scatter([(e - ref).days for e in ice_dates], y,
                       color="#3b82f6", s=55, label="ICE Endex expiry",
                       zorder=3)
            ax.scatter([(e - ref).days for e in legacy_dates], y,
                       color="#f87171", s=35, marker="D",
                       label="Legacy expiry", zorder=3)

            ax.set_yticks(y)
            ax.set_yticklabels(df["Contract"])
            ax.invert_yaxis()
            ax.set_xlabel(f"Calendar days from {ref}")
            ax.set_title("Expiries & futures LTD timeline",
                         color="#f1f5f9", fontsize=11)
            ax.grid(True, axis="x", alpha=0.3)
            ax.legend(facecolor="#161625", edgecolor="#2a2a42",
                      labelcolor="#cbd5e1", framealpha=0.9, fontsize=9,
                      loc="lower right")
            ax.set_facecolor("#0f0f1a")
            fig.patch.set_facecolor("#0f0f1a")
            plt.tight_layout()
            plt.show()


for widget in (ref_date_w, n_w):
    widget.observe(_render_expiries, names="value")

_render_expiries()

display(w.HBox([ref_date_w, n_w]), expiry_out)


---

**Tips for Section 5 :**

* The **ICE Endex** rule is the officially published one ; the
  **legacy** rule (5 business days before the futures LTD) is what all
  pricing helpers in this project currently use — the two expiries
  usually differ by **1 to 4 calendar days**.
* The *Futures LTD* is the last trading day of the **futures** contract
  (= last business day of the month before delivery). Options stop
  trading earlier than the future.
* Move the reference-date picker to project T into the past or the
  future ; the T column updates live.
* A `—` in the *Trading day ?* column would flag an expiry that fell on
  a holiday — with the ICE Endex algorithm this should never happen by
  construction.
